In [ ]:
# Import all necessary packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import rootutils

# Set directories and load data
path_root = str(rootutils.find_root(indicator=".project-root"))
path_plots = f"{path_root}/plots"
path_data = f"{path_root}/data/cytokines-regression"
df_enrichr = pd.read_excel(f"{path_data}/union_gsea.xlsx", index_col=0)

# Create dataframe for the figure
trgt_libs = {
    'GO_Biological_Process_2023': 'GO Biological Process',
    'GO_Molecular_Function_2023': 'GO Molecular Function',
    'GO_Cellular_Component_2023': 'GO Cellular Component',
    'KEGG_2021_Human': 'KEGG'
}
trgt_libs_colors = {
    'GO Biological Process': 'crimson',
    'GO Molecular Function': 'dodgerblue',
    'GO Cellular Component': 'limegreen',
    'KEGG': 'gold'
}

df_enrichr = df_enrichr.loc[(df_enrichr['Gene_set'].isin(trgt_libs.keys())) & (df_enrichr['Adjusted P-value'] < 0.05)]
for trgt_lib_id, trgt_lib in enumerate(trgt_libs):
    df_enrichr.loc[df_enrichr['Gene_set'] == trgt_lib, 'Order'] = trgt_lib_id
df_enrichr.sort_values(['Order', 'Adjusted P-value'], ascending=[True, True], inplace=True)
df_enrichr['Gene_set'].replace(trgt_libs, inplace=True)
df_enrichr.rename(columns={'Gene_set': 'Gene Set'}, inplace=True)
df_enrichr[r"$-\log_{10}(\mathrm{p-value})$"] = -np.log10(df_enrichr['Adjusted P-value'].values)
df_enrichr[['Genes In', 'Genes Max']] = df_enrichr['Overlap'].str.split('/', expand=True)
df_enrichr['% in Gene Set'] = df_enrichr['Genes In'].values.astype(float) / df_enrichr['Genes Max'].values.astype(float) * 100

# Plot Figure 3a 
sns.set_theme(style='ticks')
fig, ax = plt.subplots(figsize=(10, 4))
barplot = sns.barplot(
    data=df_enrichr,
    x=r"$-\log_{10}(\mathrm{p-value})$",
    y='Term',
    hue='Gene Set',
    palette=trgt_libs_colors,
    edgecolor='black',
    dodge=False,
    width=0.25,
    ax=ax,
)
sns.scatterplot(
    data=df_enrichr,
    x=r"$-\log_{10}(\mathrm{p-value})$",
    y='Term',
    hue='Gene Set',
    palette=trgt_libs_colors,
    edgecolor='black',
    size='% in Gene Set',
    sizes=(100, 800),
    ax=ax,
)
ax.set_ylabel('')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.savefig(f"{path_plots}/figure3a.png", bbox_inches='tight', dpi=200)
plt.savefig(f"{path_plots}/figure3a.pdf", bbox_inches='tight')
plt.close()